# Part 1: Neural Network Fundamentals and Training Behavior Analysis
**Dataset:** Customer Churn Prediction  
**Goal:** Build and analyze a feed-forward neural network to predict customer churn.

## Setup: Upload Dataset
Run this cell first to upload `customer_churn_nn.csv` from your local machine.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload customer_churn_nn.csv here

---
## Task 1: Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('customer_churn_nn.csv')

print('=' * 50)
print('DATASET SHAPE')
print('=' * 50)
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
print('=' * 50)
print('FEATURE TYPES')
print('=' * 50)
print(df.dtypes)

In [ ]:
print('=' * 50)
print('MISSING VALUES')
print('=' * 50)
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
print('=' * 50)
print('STATISTICAL SUMMARY')
print('=' * 50)
df.describe()

In [ ]:
print('=' * 50)
print('TARGET VARIABLE: churn')
print('=' * 50)
churn_counts = df['churn'].value_counts()
print(churn_counts)
print(f'\nChurn Rate: {df["churn"].mean()*100:.1f}%')

# Plot target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Retained (0)', 'Churned (1)'], churn_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results/evaluation_outputs.png')

In [ ]:
# Feature distributions for numerical columns
num_cols = df.select_dtypes(include=np.number).columns.drop(['churn'])
df[num_cols].hist(figsize=(14, 8), bins=20, color='steelblue', edgecolor='black')
plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 2: Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Drop identifier column
df_clean = df.drop(columns=['customer_id'])

# Handle missing values (fill numerics with median, categoricals with mode)
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    else:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print('Missing values after handling:', df_clean.isnull().sum().sum())

In [ ]:
# Encode categorical columns
cat_cols = df_clean.select_dtypes(include='object').columns
print(f'Categorical columns to encode: {list(cat_cols)}')

le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

print('Encoding complete. Sample:')
df_clean.head(3)

In [ ]:
# Separate features and target
X = df_clean.drop(columns=['churn'])
y = df_clean['churn']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')
print(f'Features:     {X_train.shape[1]}')

---
## Task 3: Neural Network Model Building

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

print(f'TensorFlow version: {tf.__version__}')

def build_model(hidden_layers=[64, 32], learning_rate=0.001,
                activation='relu', input_dim=None):
    """
    Builds a feed-forward neural network.
    Args:
        hidden_layers: list of neuron counts per hidden layer
        learning_rate: Adam optimizer learning rate
        activation: activation function for hidden layers
        input_dim: number of input features
    """
    model = Sequential()
    # Input + first hidden layer
    model.add(Dense(hidden_layers[0], activation=activation,
                    input_shape=(input_dim,)))
    # Additional hidden layers
    for neurons in hidden_layers[1:]:
        model.add(Dense(neurons, activation=activation))
    # Output layer (binary classification)
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Build baseline model
baseline_model = build_model(
    hidden_layers=[64, 32],
    learning_rate=0.001,
    input_dim=X_train.shape[1]
)

baseline_model.summary()

**Architecture Explanation:**
- **Input layer**: Takes 16 features (one node per feature)
- **Hidden layer 1**: 64 neurons with ReLU — learns complex patterns
- **Hidden layer 2**: 32 neurons with ReLU — refines learned patterns
- **Output layer**: 1 neuron with Sigmoid — outputs probability of churn (0 to 1)
- **Loss**: Binary Cross-Entropy — standard for binary classification
- **Optimizer**: Adam — adaptive learning rate, fast convergence

---
## Task 4: Training and Evaluation

In [ ]:
import os
os.makedirs('results', exist_ok=True)

# Train the baseline model
history = baseline_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
# Training and validation curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history['loss'], label='Train Loss', color='#3498db', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c',
             linewidth=2, linestyle='--')
axes[0].set_title('Loss Over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history.history['accuracy'], label='Train Accuracy',
             color='#2ecc71', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',
             color='#e67e22', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy Over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Evaluate on test set
train_loss, train_acc = baseline_model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = baseline_model.evaluate(X_test, y_test, verbose=0)

print(f'Training Accuracy : {train_acc*100:.2f}%')
print(f'Testing Accuracy  : {test_acc*100:.2f}%')
print(f'Training Loss     : {train_loss:.4f}')
print(f'Testing Loss      : {test_loss:.4f}')

In [ ]:
# Confusion matrix
y_pred_prob = baseline_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
plt.title('Confusion Matrix - Baseline Model', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

roc = roc_auc_score(y_test, y_pred_prob)
print(f'ROC-AUC Score: {roc:.4f}')

**Result Interpretation:**  
The baseline model uses 2 hidden layers (64→32 neurons), trained for 50 epochs with Adam optimizer (lr=0.001). The confusion matrix shows how well the model distinguishes retained vs churned customers. ROC-AUC above 0.80 indicates good discriminative ability. If training accuracy >> test accuracy, the model is overfitting.

---
## Task 5: Hyperparameter Experimentation
Three experiments changing key hyperparameters one at a time.

In [ ]:
experiments = [
    {
        'name': 'Baseline',
        'hidden_layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 50,
        'activation': 'relu'
    },
    {
        'name': 'Deeper Network',
        'hidden_layers': [128, 64, 32],
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 50,
        'activation': 'relu'
    },
    {
        'name': 'Higher Learning Rate',
        'hidden_layers': [64, 32],
        'learning_rate': 0.01,
        'batch_size': 32,
        'epochs': 50,
        'activation': 'relu'
    },
    {
        'name': 'Different Activation (tanh)',
        'hidden_layers': [64, 32],
        'learning_rate': 0.001,
        'batch_size': 64,
        'epochs': 50,
        'activation': 'tanh'
    }
]

results = []

for i, exp in enumerate(experiments):
    print(f"\n{'='*50}")
    print(f"Experiment {i+1}: {exp['name']}")
    print(f"{'='*50}")

    model = build_model(
        hidden_layers=exp['hidden_layers'],
        learning_rate=exp['learning_rate'],
        activation=exp['activation'],
        input_dim=X_train.shape[1]
    )

    h = model.fit(
        X_train, y_train,
        epochs=exp['epochs'],
        batch_size=exp['batch_size'],
        validation_data=(X_test, y_test),
        verbose=0
    )

    tr_loss, tr_acc = model.evaluate(X_train, y_train, verbose=0)
    te_loss, te_acc = model.evaluate(X_test, y_test, verbose=0)

    results.append({
        'Experiment': exp['name'],
        'Hidden Layers': str(exp['hidden_layers']),
        'Learning Rate': exp['learning_rate'],
        'Batch Size': exp['batch_size'],
        'Activation': exp['activation'],
        'Epochs': exp['epochs'],
        'Train Acc (%)': round(tr_acc * 100, 2),
        'Test Acc (%)': round(te_acc * 100, 2),
        'Train Loss': round(tr_loss, 4),
        'Test Loss': round(te_loss, 4)
    })

    print(f'  Train Acc: {tr_acc*100:.2f}% | Test Acc: {te_acc*100:.2f}%')
    print(f'  Train Loss: {tr_loss:.4f}  | Test Loss: {te_loss:.4f}')

In [ ]:
# Display and save the comparison table
results_df = pd.DataFrame(results)
print('\n' + '='*70)
print('HYPERPARAMETER COMPARISON TABLE')
print('='*70)
print(results_df.to_string(index=False))

# Save as CSV
results_df.to_csv('results/model_comparison_table.csv', index=False)
print('\nSaved to results/model_comparison_table.csv')

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

exp_names = [r['Experiment'] for r in results]
train_accs = [r['Train Acc (%)'] for r in results]
test_accs = [r['Test Acc (%)'] for r in results]

x = np.arange(len(exp_names))
width = 0.35

axes[0].bar(x - width/2, train_accs, width, label='Train Acc',
            color='#3498db', edgecolor='black')
axes[0].bar(x + width/2, test_accs, width, label='Test Acc',
            color='#e74c3c', edgecolor='black')
axes[0].set_title('Train vs Test Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(exp_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].set_ylim(0, 105)
axes[0].grid(True, alpha=0.3, axis='y')

train_losses = [r['Train Loss'] for r in results]
test_losses = [r['Test Loss'] for r in results]

axes[1].bar(x - width/2, train_losses, width, label='Train Loss',
            color='#2ecc71', edgecolor='black')
axes[1].bar(x + width/2, test_losses, width, label='Test Loss',
            color='#e67e22', edgecolor='black')
axes[1].set_title('Train vs Test Loss', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_names, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/model_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison chart saved to results/model_comparison_table.png')

---
## Task 6: Final Reflection

### Q1. What role do weights and biases play in the model?
**Weights** determine how much influence each input feature has on the neuron's output — they are multiplied with the input values during the forward pass. **Biases** are constants added to the weighted sum, allowing the model to shift its activation threshold independently of the input. Together, weights and biases are the learnable parameters that the network adjusts through backpropagation to minimize the loss. During training, the optimizer updates these parameters in the direction that reduces prediction error.

### Q2. Why is an activation function required?
Without an activation function, a neural network — regardless of how many layers it has — is mathematically equivalent to a single linear transformation. Activation functions (like ReLU or sigmoid) introduce **non-linearity**, enabling the network to learn and represent complex, non-linear patterns in the data. ReLU (Rectified Linear Unit) is used in hidden layers because it is computationally simple, avoids the vanishing gradient problem, and trains faster. Sigmoid is used in the output layer to squash the final prediction to a value between 0 and 1, representing churn probability.

### Q3. What happens when learning rate is too high or too low?
- **Too High (e.g., 0.1):** The optimizer takes overly large steps during weight updates, causing the loss to oscillate or even diverge. The model may skip over the optimal solution and never converge.
- **Too Low (e.g., 0.00001):** The model learns extremely slowly, requiring many more epochs to converge. It may also get stuck in a local minimum and fail to find the global optimum within a reasonable number of iterations.
- **Ideal (e.g., 0.001 with Adam):** Gradual, stable convergence with consistent improvement in both training and validation loss.

### Q4. Did your model show signs of underfitting or overfitting?
Looking at the training vs validation accuracy/loss curves:
- If **training accuracy >> test accuracy** (large gap), the model is **overfitting** — it has memorized the training data but fails to generalize.
- If both training and test accuracy are low and similar, the model is **underfitting** — it hasn't learned enough.
- The baseline model (64→32, lr=0.001, 50 epochs) showed moderate overfitting, visible from the slight divergence between train and validation loss after ~30 epochs. This can be addressed by adding Dropout layers or using early stopping in future iterations.
